# Kaggle Playground S6E8: Predicting Smartphone Addiction
## Attempt 1: Exploratory Data Analysis & Baseline LightGBM

### Experiment Summary
- **Model Family**: LightGBM Classifier (`LGBMClassifier`)
- **Validation Strategy**: 5-Fold Stratified Cross-Validation (`SEED = 42`)
- **Target Metric**: Out-of-Fold (OOF) ROC-AUC
- **Feature Set**: 12 Raw Features (Base Features Only)
- **Primary Goal**: Establish a reliable 5-Fold Stratified CV baseline score to validate future feature engineering and ensembling attempts.


---
### 1. Environment Setup & Global Configuration
Initialize random seed for exact reproducibility across folds (`SEED = 42`), import required packages, and define path constants.


In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATA_DIR = '../Data/Raw'
SUB_DIR = '../Data/Processed'
os.makedirs(SUB_DIR, exist_ok=True)

print(f"LightGBM Version: {lgb.__version__}")
print("Environment initialized successfully.")


---
### 2. Data Loading & Exploratory Target Inspection
Ingest `train.csv` and `test.csv`, inspect table shapes, define target (`addicted_label`) and identifier (`id`) columns, and check target distribution.


In [2]:
train_path = os.path.join(DATA_DIR, 'train.csv')
test_path = os.path.join(DATA_DIR, 'test.csv')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(f"Train Dataset Shape: {train.shape}")
print(f"Test Dataset Shape:  {test.shape}")

TARGET = 'addicted_label'
ID_COL = 'id'

feature_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]
cat_cols = ['gender', 'stress_level', 'academic_work_impact']

print(f"\nBase Features ({len(feature_cols)}): {feature_cols}")
print("\nTarget Distribution:")
print(train[TARGET].value_counts(normalize=True).map('{:.2%}'.format))


---
### 3. Categorical Preprocessing
Convert categorical string/object columns (`gender`, `stress_level`, `academic_work_impact`) to pandas `category` dtypes for LightGBM's native categorical split handling.


In [3]:
for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

print("Categorical features cast to category dtype successfully.")


---
### 4. 5-Fold Stratified Cross-Validation & LightGBM Training
Train a baseline `LGBMClassifier` using 5-Fold Stratified Cross-Validation with early stopping at 50 rounds.


In [4]:
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'n_estimators': 1500,
    'num_leaves': 31,
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

print(f"Starting 5-Fold Stratified Cross-Validation on Baseline Features...")
print("=" * 70)

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train[TARGET]), 1):
    X_train, y_train = train.iloc[train_idx][feature_cols], train.iloc[train_idx][TARGET]
    X_val, y_val = train.iloc[val_idx][feature_cols], train.iloc[val_idx][TARGET]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    test_preds += model.predict_proba(test[feature_cols])[:, 1] / skf.n_splits

    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"Fold {fold} ROC-AUC: {fold_auc:.6f} | Best Iteration: {model.best_iteration_}")

print("=" * 70)
overall_oof_auc = roc_auc_score(train[TARGET], oof_preds)
print(f"\nOverall Baseline LightGBM OOF ROC-AUC: {overall_oof_auc:.6f}")


---
### 5. Save Out-of-Fold Predictions & Export Submission
Save out-of-fold validation probabilities (`oof_lgb_attempt1.npy`) and export `submission_attempt1_lgb.csv`.


In [5]:
np.save(os.path.join(SUB_DIR, 'oof_lgb_attempt1.npy'), oof_preds)
np.save(os.path.join(SUB_DIR, 'test_lgb_attempt1.npy'), test_preds)

sub = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_preds
})

submission_path = os.path.join(SUB_DIR, 'submission_attempt1_lgb.csv')
sub.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Null values count: {sub[TARGET].isnull().sum()}")
print(f"Min prob: {sub[TARGET].min():.6f} | Max prob: {sub[TARGET].max():.6f}")
